In [ ]:
#| default_exp unigram

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json, re, mmap, numpy as np
from array import array
from tokenizers import Tokenizer, PreTokenizedString
from fastcore.all import ifnone
from laghu.core import log, find_vocab, is_unigram

## Why

`tokenizers` builds a character trie over a Unigram model's pieces. For potion-multilingual-128M's 500,353 pieces that trie is about 610 MB
of private memory in every process that loads the tokenizer. The segmentation itself is small: a Viterbi pass over each pre-token,
scoring every substring that is a piece. Here the pieces sit in a dict (about 60 MB for the same 500k), the longest piece starting with
each pair of letters bounds the substrings tried, and the rest of the pipeline (normalizer, pre-tokenizer) is `tokenizers`' own, built
from the same tokenizer.json with the vocabulary taken out.

## Reading the vocabulary

The pieces are read one at a time off the mapped file. Parsing the whole tokenizer.json as JSON, or reading it into a bytes object
first, leaves about 250 MB that the allocator keeps after the parse.

In [ ]:
#| export
_ENTRY = re.compile(rb'\s*\[\s*"((?:[^"\\]|\\.)*)"\s*,\s*([^\],\s]+)\s*\]\s*(,?)')
_CLOSE = re.compile(rb'\s*\]')

def read_vocab(fn):
    "Unigram tokenizer.json `fn` as (piece → id, scores, the JSON with the vocabulary emptied); a repeated piece keeps its last id, as in `tokenizers`."
    with open(fn, 'rb') as f, mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ) as t:
        v0, br = find_vocab(t)
        if br != b'[': raise NotImplementedError(f'{fn}: not a Unigram vocabulary')
        ids, scores, at, i = {}, array('d'), v0, 0
        while m := _ENTRY.match(t, at):
            p = m.group(1); ids[json.loads(b'"' + p + b'"') if b'\\' in p else p.decode()] = i; scores.append(float(m.group(2))); i += 1; at = m.end()
            if not m.group(3): break
        if not (c := _CLOSE.match(t, at)): raise ValueError(f'{fn}: vocabulary not read to its end (stopped at byte {at})')
        return ids, scores, json.loads(t[:v0] + t[c.start():])

## Segmenting a pre-token

`viterbi` is `Unigram::encode_optimized` from HF tokenizers. Each position keeps the best-scoring path ending there; a later candidate
replaces it only when strictly better, so on a tie the path found first stays. A character that starts no one-character piece is an
unknown piece scored `min_score - 10`. Walking back, a run of unknown pieces becomes one string, looked up as a piece (it is the unknown
id unless the run happens to spell a piece), which is what `fuse_unk` does.

In [ ]:
#| export
UNK_PENALTY = 10.0

def _unk(p, ids, unk_id, byte_fallback):
    "The ids of an unknown run `p`: a piece if it spells one, else its UTF-8 bytes as `<0xXX>` pieces under byte fallback, else the unknown id."
    if (k := ids.get(p)) is not None: return [k]
    if byte_fallback and None not in (bs := [ids.get(f'<0x{b:02X}>') for b in p.encode()]): return bs
    if unk_id < 0: raise ValueError(f'Encountered an unknown token but `unk_id` is missing: {p!r}')   # tokenizers' error, and model2vec's
    return [unk_id]

def viterbi(s, ids, scores, reach, unk_id, unk_score, byte_fallback=False):
    "The ids of the best segmentation of `s` into pieces of `ids`, unknown runs fused."
    n = len(s)
    best, start, tid = [0.0] + [None] * n, [0] * (n + 1), [0] * (n + 1)
    for i in range(n):
        b, single = best[i], False
        for j in range(i + 1, min(n, i + reach.get(s[i:i + 2], 1)) + 1):
            k = ids.get(s[i:j])
            if k is None: continue
            c = scores[k] + b
            if best[j] is None or c > best[j]: best[j], start[j], tid[j] = c, i, k
            if j == i + 1: single = True
        if not single:
            c = unk_score + b
            if best[i + 1] is None or c > best[i + 1]: best[i + 1], start[i + 1], tid[i + 1] = c, i, unk_id
    out, e, run = [], n, None
    while e > 0:
        if tid[e] == unk_id: run = run or e
        else:
            if run: out += _unk(s[e:run], ids, unk_id, byte_fallback)[::-1]; run = None
            out.append(tid[e])
        e = start[e]
        if run and e == 0: out += _unk(s[:run], ids, unk_id, byte_fallback)[::-1]
    return out[::-1]

In [ ]:
ids = {'a': 0, 'b': 1, 'ab': 2, '<unk>': 3, 'xy': 4}
sc = array('d', [-1.0, -1.0, -1.5, -5.0, -2.0])
reach = dict(ab=2, xy=2)
v = lambda s: viterbi(s, ids, sc, reach, 3, min(sc) - UNK_PENALTY)
assert v('ab') == [2] and v('ba') == [1, 0] and v('') == []
assert v('aqqb') == [0, 3, 1]          # the two unknown letters are one token
assert v('xy') == [4] and v('xq') == [3] and v('qxy') == [3, 4]
bids = {**ids, '<0x71>': 5, '<0xC3>': 6, '<0xA9>': 7}
bv = lambda s: viterbi(s, bids, array('d', [*sc, -1, -1, -1]), reach, 3, min(sc) - UNK_PENALTY, True)
assert bv('aqqb') == [0, 5, 5, 1] and bv('é') == [6, 7] and bv('ä') == [3]   # 'ä' is 0xC3 0xA4, and there is no <0xA4>

## The tokenizer

`Unigram` reproduces `Tokenizer.encode(text, add_special_tokens=False).ids` for the tokenizer.json it is given:

1. added tokens that are not `normalized` (`[PAD]`, `<extra_id_0>`, XLM-R's `<mask>`…) are cut out of the raw text, longest match first;
   `lstrip` and `rstrip` make a match take the whitespace beside it;
2. each stretch between them goes through the normalizer, and `normalized` added tokens, themselves normalized first (one model's `[PAD]`
   becomes `[ PAD ]`), are cut out of the normalized text the same way;
3. what is left goes through the pre-tokenizer, and each pre-token is segmented by `viterbi`;
4. truncation and padding, when tokenizer.json sets them, are applied to each batch as `encode_batch` applies them: model2vec embeds the pad ids.

A Metaspace pre-tokenizer with `prepend_scheme: first` adds `▁` only where the normalized text begins at offset 0 of the original: a
leading space or byte-order mark that the normalizer drops means no `▁`. So the opening stretch goes through a `PreTokenizedString`,
which keeps the alignments, and the stretches after an added token get a copy of the pre-tokenizer with the scheme set to `never`.

What it does not reproduce raises `NotImplementedError`, and the caller falls back to `tokenizers`: `single_word` added
tokens (Rust's word test is Unicode `Alphabetic`, which Python does not expose), and `normalized` added tokens under `prepend_scheme: first`.

In [ ]:
#| export
WS = frozenset('\t\n\x0b\x0c\r \x85\xa0\u1680\u2000\u2001\u2002\u2003\u2004\u2005\u2006\u2007\u2008\u2009\u200a\u2028\u2029\u202f\u205f\u3000')   # Rust's char::is_whitespace

def _metaspace(pt, scheme):
    "Pre-tokenizer JSON `pt` with every Metaspace's `prepend_scheme: first` set to `scheme`."
    if isinstance(pt, list): return [_metaspace(o, scheme) for o in pt]
    if not isinstance(pt, dict): return pt
    pt = {k: _metaspace(v, scheme) for k, v in pt.items()}
    if pt.get('type') == 'Metaspace' and pt.get('prepend_scheme') == 'first': pt['prepend_scheme'] = scheme
    return pt

def _pipeline(d, pre):
    "The normalizer and pre-tokenizer of tokenizer JSON `d`, with pre-tokenizer `pre`, and nothing else of it."
    d = dict(d, pre_tokenizer=pre, added_tokens=[], post_processor=None, decoder=None, truncation=None, padding=None)
    d['model'] = dict(d['model'], vocab=[['<unk>', 0.0]], unk_id=0, byte_fallback=False)
    t = Tokenizer.from_str(json.dumps(d))
    return t.normalizer, t.pre_tokenizer, t

class Added:
    "A set of added tokens found in a string as `tokenizers`' `AddedVocabulary.find_matches` finds them."
    def __init__(self, toks, norm=None):
        "`toks` are matched as written, or as `norm` normalizes them when it is given (`normalized` tokens)."
        self.toks = {(norm.normalize_str(o['content']) if norm else o['content']): (o['id'], o.get('lstrip'), o.get('rstrip')) for o in toks}
        self.toks.pop('', None)
        self.rx = re.compile('|'.join(map(re.escape, sorted(self.toks, key=len, reverse=True)))) if self.toks else None

    def split(self, s):
        "`s` cut into `(text, None)` and `(content, id)` parts, in order, empty text dropped."
        if not self.rx: return [(s, None)] if s else []
        out, prev = [], 0
        for m in self.rx.finditer(s):
            st, sp = m.span(); k, ls, rs = self.toks[m.group()]
            if ls:
                while st > prev and s[st - 1] in WS: st -= 1
            if rs:
                while sp < len(s) and s[sp] in WS: sp += 1
            if st > prev: out.append((s[prev:st], None))
            out.append((s[st:sp], k)); prev = sp
        if prev < len(s): out.append((s[prev:], None))
        return out

class Unigram:
    "A Unigram tokenizer.json's `encode_batch(texts, add_special_tokens=False)` ids, with no trie."
    def __init__(self, fn):
        self.ids, self.scores, d = read_vocab(fn)
        mdl = d['model']
        if mdl.get('type', 'Unigram') != 'Unigram': raise NotImplementedError(f"{fn}: model is {mdl.get('type')}")
        added = d.get('added_tokens', [])
        if any(o.get('single_word') for o in added): raise NotImplementedError(f'{fn}: single_word added tokens')
        self.byte_fallback, self.unk_id = bool(mdl.get('byte_fallback')), ifnone(mdl.get('unk_id'), -1)   # -1: none, an error if met
        self.trunc, self.pad = d.get('truncation'), d.get('padding')
        self.unk_score = min(self.scores) - UNK_PENALTY
        self.reach = {}   # a piece's first two letters → the longest piece that starts so: no longer substring is tried
        for w in self.ids:
            if len(w) > 1 and len(w) > self.reach.get(w[:2], 0): self.reach[w[:2]] = len(w)
        self.added = {o['content']: o['id'] for o in added}
        self.norm, self.pre, t = _pipeline(d, d.get('pre_tokenizer'))
        self.raw, self.nrm = Added([o for o in added if not o.get('normalized')]), Added([o for o in added if o.get('normalized')], self.norm)
        later = _metaspace(d.get('pre_tokenizer'), 'never')
        self.pre_later = self.pre if later == d.get('pre_tokenizer') else _pipeline(d, later)[1]
        if self.nrm.rx and self.pre_later is not self.pre: raise NotImplementedError(f'{fn}: normalized added tokens under prepend_scheme first')
        self.model_unk = getattr(t.model, 'unk_token', None)   # model2vec drops this token's id; a Unigram model has none
        extra = [k for k in self.added if k not in self.ids]
        self.n = len(self.ids) + len(extra)   # the size of `Tokenizer.get_vocab()`
        lens = np.fromiter((len(k) for k in [*self.ids, *extra]), np.int32, self.n)
        self.median = int(np.median(lens)) if self.n else 0

    def _first(self, s):
        "The pre-tokens of the stretch opening the text, offsets kept, so `first` prepends only where the normalized text starts at 0."
        pts = PreTokenizedString(s)
        if self.norm: pts.normalize(self.norm.normalize)
        pts.split(lambda i, ns: [ns[0:len(ns.normalized)]] if ns.normalized else [])   # as tokenizers re-slices after its added-token pass
        self.pre.pre_tokenize(pts)
        return [p for p, *_ in pts.get_splits()]

    def _pieces(self, nz, pt):
        out = []
        for p, k in self.nrm.split(nz):
            if k is not None: out.append(k); continue
            for q in ([q for q, _ in pt.pre_tokenize_str(p)] if pt else [p]):
                out += viterbi(q, self.ids, self.scores, self.reach, self.unk_id, self.unk_score, self.byte_fallback)
        return out

    def _seg(self, s, first):
        if first and self.pre_later is not self.pre:
            return [k for p in self._first(s) for k in viterbi(p, self.ids, self.scores, self.reach, self.unk_id, self.unk_score, self.byte_fallback)]
        nz = self.norm.normalize_str(s) if self.norm else s
        return self._pieces(nz, self.pre if first else self.pre_later) if nz else []

    def encode(self, s):
        "The ids of text `s`, before truncation and padding."
        out, at = [], 0
        for p, k in self.raw.split(s):
            out += self._seg(p, at == 0) if k is None else [k]; at += len(p)
        return out

    def __call__(self, txts):
        "The ids of each of `txts`, as one batch: truncated and padded when tokenizer.json says so."
        out = [self.encode(t) for t in txts]
        if t := self.trunc:
            L, left = t['max_length'], t.get('direction') == 'Left'
            out = [(o[-L:] if L else []) if left else o[:L] for o in out]
        if (p := self.pad) and out:
            st = p.get('strategy')
            n = max(map(len, out)) if st == 'BatchLongest' else (st or {}).get('Fixed', 0)
            if (m := p.get('pad_to_multiple_of')) and n % m: n += m - n % m
            pid, left = p.get('pad_id', 0), p.get('direction') == 'Left'
            out = [([pid] * (n - len(o)) + o if left else o + [pid] * (n - len(o))) if len(o) < n else o for o in out]
        return out

    @property
    def unk_token_id(self):
        "The id model2vec drops from every encoding, or None."
        return None if self.model_unk is None else self.vocab()[self.model_unk]

    def vocab(self):
        "`Tokenizer.get_vocab()`: piece → id, the added tokens over the model's."
        return {**self.ids, **self.added}

## Everything else

`HFTok` is `tokenizers` itself, used for WordPiece, BPE and WordLevel models and for whatever `Unigram` refuses. Its attributes are the
ones model2vec reads off the tokenizer, so `LeanModel` treats the two alike.

In [ ]:
#| export
class HFTok:
    "`tokenizers`' own encoding, as model2vec calls it."
    def __init__(self, fn):
        self.tok = Tokenizer.from_file(str(fn))
        v = self.tok.get_vocab()
        self.n, self.median = len(v), int(np.median([len(o) for o in v])) if v else 0
        unk = getattr(self.tok.model, 'unk_token', None)
        self.unk_token_id = v[unk] if unk is not None else None
        self._enc = getattr(self.tok, 'encode_batch_fast', self.tok.encode_batch)

    def __call__(self, txts):
        "The ids of each of `txts`, one batch as `tokenizers` encodes it (padding, if the file asks for it, is per batch)."
        return [e.ids for e in self._enc(list(txts), add_special_tokens=False)]

    def vocab(self): return self.tok.get_vocab()

def tokenizer(fn, lean=True):
    "A `Unigram` for a Unigram tokenizer.json `fn` when `lean` and it can be done, else an `HFTok`."
    if lean and is_unigram(fn):
        try: return Unigram(fn)
        except NotImplementedError as e: log.info(f'laghu: {e}; tokenizing with tokenizers')
    return HFTok(fn)

## Tests

Against `tokenizers`' `encode_batch` on the corpus, in batches of 64 so that padding to the longest is exercised, for every Unigram
tokenizer at hand: potion-multilingual-128M's (bge-m3's, Metaspace `always`, two added tokens), t5-small's (an old file with no model
type, WhitespaceSplit before Metaspace, 103 added tokens), the tokenizers of five published distills that each bring something else,
and variants of the first written here.

In [ ]:
from laghu.core import try_resolve, model_files, corpus, ODD
from fastcore.all import Path
import tempfile

def same(fn, txts, bs=64):
    "Whether `Unigram(fn)` gives `tokenizers`' ids on every one of `txts`; the first difference is printed."
    u, t = Unigram(fn), Tokenizer.from_file(str(fn))
    for i in range(0, len(txts), bs):
        b = txts[i:i + bs]
        for s, a, e in zip(b, u(b), t.encode_batch(b, add_special_tokens=False)):
            if a != e.ids: print(repr(s[:80]), a[:20], e.ids[:20]); return False
    return True

def hub_file(repo, fn):
    "One file of `repo` from the Hub cache or downloaded, or None offline."
    from huggingface_hub import hf_hub_download
    try: return Path(hf_hub_download(repo, fn))
    except Exception as e: print(f'skipping {repo}: {e!r}')

TOKS = ['<mask>', '<s>', '</s>', '<pad>', '[PAD]', '[UNK]', '<unk>', '<extra_id_0>', '<bos>', '<eos>', 'Computer Science']
txts = corpus(1500) + [f'{a}{t}{b}' for t in TOKS for a in ('', 'x', 'x ', ' x  ', '\u3000') for b in ('', 'y', ' y', '  \n')]
tmp = Path(tempfile.mkdtemp())

In [ ]:
if f := try_resolve('minishlab/potion-multilingual-128M'):
    fn = model_files(f).tok
    assert same(fn, txts)

In [ ]:
if fn := hub_file('google-t5/t5-small', 'tokenizer.json'):
    assert is_unigram(fn) and 'type' not in json.loads(fn.read_text())['model']
    assert same(fn, txts)

The published tokenizers: jina-embeddings-v3 (XLM-R's: `<mask>` with `lstrip`, WhitespaceSplit and Metaspace), the EmbeddingGemma
distill (byte fallback, a Replace normalizer, 255k pieces), a Spanish intent model (ByteLevel pre-tokenizer, `normalized` added
tokens, truncation at 512, padding fixed at 512), the Qwen3 distill (ByteLevel over a Unigram vocabulary), and a Danish model.

In [ ]:
for repo in ['CISCai/jina-embeddings-v3-separation-distilled', 'mikeee/m2v-gemma-embedding-300m', 'OpenVoiceOS/ovos-m2v-intents-es-ES-bne-v5',
             'tss-deposium/m2v-qwen3-embedding-0.6b-1024d', 'andersborges/model2vecdk']:
    if fn := hub_file(repo, 'tokenizer.json'): assert isinstance(tokenizer(fn), Unigram), repo; assert same(fn, txts), repo

Variants of potion-multilingual-128M's file, each changing one thing: `prepend_scheme` `first` and `never`; no normalizer; no
pre-tokenizer; `[PAD]` with `lstrip`, `[UNK]` with `rstrip`; both `normalized`; byte fallback over pieces `<0x00>`…`<0xFF>`; padding to
the longest, on the left, to a multiple of 8; truncation at 16 from the left.

In [ ]:
def variant(nm, ch):
    d2 = json.loads(json.dumps(d)); ch(d2); (tmp/f'{nm}.json').write_text(json.dumps(d2, ensure_ascii=False)); return tmp/f'{nm}.json'

def bytes_(d):
    v = d['model']['vocab']; v += [[f'<0x{b:02X}>', -20.0] for b in range(256)]; d['model']['byte_fallback'] = True

def flag(**kw):
    def f(d): [o.update(kw.get(o['content'], {})) for o in d['added_tokens']]
    return f

if f := try_resolve('minishlab/potion-multilingual-128M'):
    d = json.loads(Path(model_files(f).tok).read_text())
    for nm, ch in [('first', lambda d: d['pre_tokenizer'].update(prepend_scheme='first')),
                   ('never', lambda d: d['pre_tokenizer'].update(prepend_scheme='never')),
                   ('nonorm', lambda d: d.update(normalizer=None)), ('nopre', lambda d: d.update(pre_tokenizer=None)),
                   ('strip', flag(**{'[PAD]': dict(lstrip=True), '[UNK]': dict(rstrip=True)})),
                   ('normed', flag(**{'[PAD]': dict(normalized=True), '[UNK]': dict(normalized=True, lstrip=True)})),
                   ('bytes', bytes_),
                   ('pad', lambda d: d.update(padding=dict(strategy='BatchLongest', direction='Left', pad_to_multiple_of=8, pad_id=0, pad_type_id=0, pad_token='[PAD]'))),
                   ('trunc', lambda d: d.update(truncation=dict(direction='Left', max_length=16, strategy='LongestFirst', stride=0)))]:
        assert same(variant(nm, ch), txts), nm

The fallback: a WordPiece file is not Unigram, and a Unigram file with a `single_word` added token is refused, so both tokenize with `tokenizers`.

In [ ]:
if f := try_resolve('minishlab/potion-base-8M'):
    assert isinstance(tokenizer(model_files(f).tok), HFTok)
if f := try_resolve('minishlab/potion-multilingual-128M'):
    assert isinstance(tokenizer(variant('sw', flag(**{'[PAD]': dict(single_word=True)}))), HFTok) and isinstance(tokenizer(model_files(f).tok), Unigram)